# Phase 3 — Per-sentence feature extraction

Builds `features/sentence_features.csv`: **one row per unique `sentence_id`**, over *all* master rows
(train + val + test), then joins onto `preds/all_predictions_long.csv` and asserts integrity.

CPU-only, Drive-backed. Nothing here touches a GPU or a tokenizer.

**Inputs**  `data/sa_master.parquet` (parquet, *not* the CSV — `tokens`/`lang_tags` must be real lists)
**Outputs** `features/sentence_features.csv`, `features/predictions_with_features.csv`, `features/feature_spec.json`

---

## The two design decisions, pinned

**1. `oov_rate` reference vocab → `wordfreq` top-20,000 English word list.**
Model-agnostic (one value per `sentence_id`), deterministic, and reproducible from a pinned package
version — which NLTK's `words` corpus is less good at. `oov_rate` is computed over **alphabetic tokens
only**, so URLs, `@mentions`, `#hashtags`, emoji and punctuation are excluded from both numerator and
denominator (they are not evidence about lexical canonicity).

*Why 20k and not 50k.* `wordfreq`'s English list is web-derived, so common romanized Hindi leaks into
it as you go deeper — measured on this package: `hai` at rank ~13.5k, `bhai` ~24k, `tum` ~27k,
`kar` ~28k, `kya` ~41k, `yaar` ~67k, `nahi` ~68k. At top-50k, eight of a twelve-word romanized-Hindi
probe list are already "in vocabulary", which would make `oov_rate` blind to exactly the tokens it is
supposed to catch. At top-20k only `hai` leaks (`log` is a real English word). The cost of the smaller
list is that rare *genuine* English gets counted as OOV — acceptable here, since tweet vocabulary sits
overwhelmingly in the high-frequency band. `OOV_VOCAB_TOP_N` is a one-line change if you want to test
sensitivity, and §3 runs a **leakage audit against your actual corpus** so the number goes in the
write-up rather than being asserted.

> **Stated confound (carry this into Phase 4):** romanized Hindi, Hinglish slang and misspellings all
> count as OOV, so `oov_rate` measures *non-canonical-English density* and will correlate strongly with
> `hi_ratio`. This is accepted, not accidental. Phase 4 **must** check collinearity/VIF before putting
> `oov_rate` and `hi_ratio` in the same regression — the handoff's model
> (`correct ~ cmi + switch_density + length + oov_rate`) deliberately omits `hi_ratio` for this reason.

**2. `switch_density` normalizer → `switches / max(1, n_lang_tokens − 1)`.**
Transitions per *adjacent language-token pair*, i.e. the share of language-token boundaries that are
actually switch points. This makes it a true rate in [0, 1] and keeps it from being mechanically
deflated by long runs of `O` tokens (URLs, mentions, emoji), which are frequent in tweets and carry no
language signal. Dividing by raw `length` would entangle the feature with `length`, which Phase 4 is
already controlling for separately.

Both are single-constant switches in the config cell (`SWITCH_DENSITY_NORM`, `OOV_VOCAB_SOURCE`) and
are written into `feature_spec.json` so the choice travels with the artifact into the Phase 6 repo.

## 0. Setup — Drive, paths

In [1]:
import os, json, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/codemix_project'
except ImportError:
    # non-Colab fallback (local testing); override with CODEMIX_PROJECT env var
    PROJECT = os.environ.get('CODEMIX_PROJECT', './codemix_project')

DATA     = os.path.join(PROJECT, 'data')
PREDS    = os.path.join(PROJECT, 'preds')
FEATURES = os.path.join(PROJECT, 'features')
os.makedirs(FEATURES, exist_ok=True)

MASTER_PARQUET = os.path.join(DATA, 'sa_master.parquet')
LABEL_MAP_JSON = os.path.join(DATA, 'label_mapping.json')
PREDS_LONG     = os.path.join(PREDS, 'all_predictions_long.csv')

FEATURES_CSV = os.path.join(FEATURES, 'sentence_features.csv')
MERGED_CSV   = os.path.join(FEATURES, 'predictions_with_features.csv')
SPEC_JSON    = os.path.join(FEATURES, 'feature_spec.json')

print('project        :', PROJECT)
print('master parquet :', os.path.exists(MASTER_PARQUET), MASTER_PARQUET)
print('label mapping  :', os.path.exists(LABEL_MAP_JSON))
print('preds long     :', os.path.exists(PREDS_LONG))
if not os.path.exists(MASTER_PARQUET):
    raise FileNotFoundError('sa_master.parquet not found — Phase 3 cannot run without it.')

Mounted at /content/drive
project        : /content/drive/MyDrive/codemix_project
master parquet : True /content/drive/MyDrive/codemix_project/data/sa_master.parquet
label mapping  : True
preds long     : True


## 1. Config — the pinned decisions live here

In [ ]:
# ---- PINNED DESIGN DECISIONS (change here, nowhere else) --------------------
SWITCH_DENSITY_NORM = 'lang_pairs'   # 'lang_pairs' -> / max(1, n_lang_tokens - 1)   [PINNED]
                                     # 'length'     -> / max(1, len(tokens))
OOV_VOCAB_SOURCE    = 'wordfreq'     # 'wordfreq' [PINNED]  |  'nltk'
OOV_VOCAB_TOP_N     = 20_000         # [PINNED] only used when OOV_VOCAB_SOURCE == 'wordfreq'
                                     # deeper lists leak romanized Hindi (kya ~41k, nahi ~68k) — see §3 audit
MIN_CAPS_LEN        = 2              # ALL-CAPS token must be longer than 1 char

LANG_LABELS = {'HI', 'EN'}           # [FIXED] language-bearing tags; 'OTHER' is dropped.
                                      # was {'Hin', 'Eng'} -- master's raw lang_tags use 'HI'/'EN'/'OTHER'
                                      # (confirmed via master['lang_tags'].iloc[0] in cell 7), so nothing
                                      # ever matched and n_lang_tokens was 0 for every single sentence.
                                      # Before trusting this, run:
                                      #   sorted(set(t for tags in master['lang_tags'] for t in tags))
                                      # to confirm there's no 4th tag hiding in the corpus.

FEATURE_COLS = ['cmi', 'length', 'switch_density', 'hi_ratio', 'caps_ratio', 'oov_rate']
DIAG_COLS    = ['n_lang_tokens', 'n_switches', 'n_alpha_tokens']   # diagnostics, not for modelling

In [6]:
%pip install -q wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 17.8 MB/s eta 0:00:00


## 2. Load the master table + structural validation

Reads the **parquet**. If `tokens` comes back as strings, you have read the CSV by mistake and every
downstream feature would be silently wrong — so that is asserted, not assumed.

In [3]:
import pandas as pd
import numpy as np

master = pd.read_parquet(MASTER_PARQUET)
print('master shape :', master.shape)
print('columns      :', master.columns.tolist())
master.head(3)

master shape : (15131, 8)
columns      : ['sentence_id', 'label', 'tokens', 'lang_tags', 'text', 'label_id', 'split', 'cmi']


,sentence_id,label,tokens,lang_tags,text,label_id,split,cmi
0,43737,negative,"[RT, @, Nehr, _, who, Sala, aate, hi, Kaam, ka...","[EN, OTHER, EN, OTHER, EN, HI, HI, HI, HI, HI,...",RT @ Nehr _ who Sala aate hi Kaam karne lag ga...,0,train,17.86
1,13353,negative,"[@, iVeenaKhan, Tum, tu, whore, ho, ., Lifafe,...","[OTHER, HI, HI, HI, HI, HI, OTHER, HI, HI, HI,...",@ iVeenaKhan Tum tu whore ho . Lifafe khud let...,0,train,20.00
2,4845,negative,"[@, bilalsoomro, _, Ye, parchi, or, selfish, n...","[OTHER, HI, OTHER, HI, HI, EN, EN, EN, HI, OTH...",@ bilalsoomro _ Ye parchi or selfish nai chala...,0,train,38.46


In [4]:
required = {'sentence_id', 'tokens', 'lang_tags', 'cmi', 'split', 'text'}
missing = required - set(master.columns)
assert not missing, f'master is missing columns: {missing}'

# parquet list columns can arrive as numpy arrays -> normalise to plain lists
master['tokens']    = master['tokens'].map(list)
master['lang_tags'] = master['lang_tags'].map(list)

# guard against having read the CSV (where these are JSON strings)
first_tok = master['tokens'].iloc[0]
assert isinstance(first_tok, list) and (len(first_tok) == 0 or isinstance(first_tok[0], str)), (
    'tokens are not list[str] — did you read sa_master.csv instead of the parquet?')

assert master['sentence_id'].is_unique, 'sentence_id is NOT unique in the master'
assert master['cmi'].notna().all(), 'master has NaN cmi'

len_mismatch = int((master['tokens'].map(len) != master['lang_tags'].map(len)).sum())
empty_tokens = int((master['tokens'].map(len) == 0).sum())

print('rows                    :', len(master))
print('unique sentence_id      :', master['sentence_id'].nunique())
print('sentence_id dtype       :', master['sentence_id'].dtype)
print('tokens/lang_tags length mismatch rows :', len_mismatch, '(features zip to the shorter of the two)')
print('empty-token rows        :', empty_tokens)
print()
print(master['split'].value_counts().to_string())
print()
print('cmi  mean/median/max    : %.3f / %.3f / %.3f' % (master.cmi.mean(), master.cmi.median(), master.cmi.max()))

rows                    : 15131
unique sentence_id      : 15131
sentence_id dtype       : object
tokens/lang_tags length mismatch rows : 0 (features zip to the shorter of the two)
empty-token rows        : 0

split
train    12104
test      1514
val       1513

cmi  mean/median/max    : 29.751 / 28.570 / 50.000


## 3. Reference vocabulary for `oov_rate`

Model-agnostic by construction: one vocabulary, applied identically to every sentence, so `oov_rate`
is a property of the *sentence*, never of a tokenizer.

In [7]:
def build_english_vocab(source: str, top_n: int):
    """Return (vocab_set, provenance_dict). Lowercased, alphabetic-only entries."""
    if source == 'wordfreq':
        try:
            from wordfreq import top_n_list
        except ImportError:
            raise ImportError("wordfreq is not installed — run the '%pip install -q wordfreq' cell above.")
        try:
            from importlib.metadata import version
            ver = version('wordfreq')
        except Exception:
            ver = 'unknown'
        raw = top_n_list('en', top_n)
        prov = {'source': 'wordfreq.top_n_list("en", %d)' % top_n, 'package_version': ver}
    elif source == 'nltk':
        try:
            import nltk
        except ImportError:
            raise ImportError("OOV_VOCAB_SOURCE='nltk' needs nltk — run: %pip install -q nltk")
        nltk.download('words', quiet=True)
        from nltk.corpus import words as nltk_words
        raw = nltk_words.words()
        prov = {'source': 'nltk.corpus.words', 'package_version': getattr(nltk, '__version__', 'unknown')}
    else:
        raise ValueError('unknown OOV_VOCAB_SOURCE: %r' % source)

    vocab = {w.lower() for w in raw if w.isalpha()}
    prov['vocab_size'] = len(vocab)
    return vocab, prov


VOCAB, VOCAB_PROV = build_english_vocab(OOV_VOCAB_SOURCE, OOV_VOCAB_TOP_N)
print(VOCAB_PROV)
probe = ['kya', 'bhai', 'nahi', 'accha', 'yaar', 'hai', 'bohot', 'tum', 'kar', 'matlab', 'sab', 'log']
print('probe caught as OOV  :', [w for w in probe if w not in VOCAB])
print('probe leaking into vocab:', [w for w in probe if w in VOCAB])

{'source': 'wordfreq.top_n_list("en", 20000)', 'package_version': '3.1.1', 'vocab_size': 19583}
probe caught as OOV  : ['kya', 'bhai', 'nahi', 'accha', 'yaar', 'bohot', 'tum', 'kar', 'matlab', 'sab']
probe leaking into vocab: ['hai', 'log']


### 3b. Vocab leakage audit (run this, quote the number in the write-up)

`oov_rate` is only meaningful if the reference list actually excludes romanized Hindi. This measures
that directly against the corpus, using the gold `lang_tags`:

- **leakage** = share of `Hin`-tagged alphabetic tokens that the "English" vocab wrongly contains
  (lower is better — these are OOV signals the feature will miss)
- **coverage** = share of `Eng`-tagged alphabetic tokens the vocab contains
  (higher is better — the complement is genuine English counted as OOV, i.e. noise)

If leakage is high, drop `OOV_VOCAB_TOP_N`; if coverage is low, raise it. Report both numbers in Phase 6.

In [ ]:
from collections import Counter

hin_tok, eng_tok = Counter(), Counter()
for toks, tags in zip(master['tokens'], master['lang_tags']):
    for tk, tg in zip(toks, tags):
        if not tk.isalpha():
            continue
        if tg == 'HI':  # [FIXED] was 'Hin' -- see LANG_LABELS fix in cell 4
            hin_tok[tk.lower()] += 1
        elif tg == 'EN':  # [FIXED] was 'Eng'
            eng_tok[tk.lower()] += 1

def _rate(counter, member):
    tot = sum(counter.values())
    hit = sum(c for w, c in counter.items() if (w in VOCAB) == member)
    return (hit / tot if tot else 0.0), tot

leak_tok, n_hin_tok = _rate(hin_tok, True)      # Hin tokens wrongly IN the English vocab
cov_tok,  n_eng_tok = _rate(eng_tok, True)      # Eng tokens correctly IN the English vocab
leak_typ = sum(1 for w in hin_tok if w in VOCAB) / max(1, len(hin_tok))
cov_typ  = sum(1 for w in eng_tok if w in VOCAB) / max(1, len(eng_tok))

VOCAB_AUDIT = {
    'hin_leakage_token_level': round(leak_tok, 4),
    'hin_leakage_type_level':  round(leak_typ, 4),
    'eng_coverage_token_level': round(cov_tok, 4),
    'eng_coverage_type_level':  round(cov_typ, 4),
    'n_hin_alpha_tokens': n_hin_tok,
    'n_eng_alpha_tokens': n_eng_tok,
}
for k, v in VOCAB_AUDIT.items():
    print('%-26s %s' % (k, v))

print()
print('top Hin-tagged types leaking into the English vocab (these will NOT count as OOV):')
leaking = [(w, c) for w, c in hin_tok.most_common() if w in VOCAB][:15]
print('   ', leaking)
print()
print('top Eng-tagged types missing from the vocab (these WILL count as OOV — noise):')
missing = [(w, c) for w, c in eng_tok.most_common() if w not in VOCAB][:15]
print('   ', missing)

## 4. Feature functions

Every feature is defined on a guarded denominator, so a tweet with zero language tokens or zero
alphabetic tokens yields `0.0` rather than a NaN. That is what makes the "no NaN" integrity gate
meaningful instead of decorative.

In [ ]:
def sentence_features(tokens, lang_tags, vocab):
    """Per-sentence features from a token list + aligned language-tag list."""
    n = len(tokens)
    m = min(n, len(lang_tags))                      # tolerate ragged rows by zipping the shorter

    # [FIXED] The previous "Title Case" normalization (.capitalize()) solved a case-
    # sensitivity problem that didn't exist -- the real mismatch was that master's raw
    # tags are 'HI'/'EN'/'OTHER' (see cell 7), not 'Hin'/'Eng'/'O'. capitalize() turns
    # 'HI' -> 'Hi' and 'EN' -> 'En', neither of which is in LANG_LABELS either, so
    # n_lang_tokens was silently 0 for all 15,131 rows and switch_density/hi_ratio were
    # constant 0.0 across the whole corpus. Normalizing to uppercase instead, to match
    # LANG_LABELS = {'HI', 'EN'} while still tolerating stray whitespace/casing.
    norm_tags = [t.strip().upper() for t in lang_tags[:m]]

    # --- language-bearing tokens (drop 'OTHER': URLs, mentions, emoji, punctuation) ---
    langs   = [t for t in norm_tags if t in LANG_LABELS]
    n_lang  = len(langs)
    n_switch = sum(1 for a, b in zip(langs, langs[1:]) if a != b)

    if SWITCH_DENSITY_NORM == 'lang_pairs':
        switch_density = n_switch / max(1, n_lang - 1)
    elif SWITCH_DENSITY_NORM == 'length':
        switch_density = n_switch / max(1, n)
    else:
        raise ValueError('unknown SWITCH_DENSITY_NORM: %r' % SWITCH_DENSITY_NORM)

    n_hin = langs.count('HI')  # [FIXED] was 'Hin'
    n_eng = langs.count('EN')  # [FIXED] was 'Eng'
    hi_ratio = n_hin / max(1, n_hin + n_eng)

    # --- alphabetic tokens only, for the surface-form features ---
    alpha   = [t for t in tokens if t.isalpha()]
    n_alpha = len(alpha)

    n_caps = sum(1 for t in alpha if len(t) >= MIN_CAPS_LEN and t.isupper())
    caps_ratio = n_caps / n_alpha if n_alpha else 0.0

    n_oov = sum(1 for t in alpha if t.lower() not in vocab)
    oov_rate = n_oov / n_alpha if n_alpha else 0.0

    return {
        'length':         n,
        'switch_density': switch_density,
        'hi_ratio':       hi_ratio,
        'caps_ratio':     caps_ratio,
        'oov_rate':       oov_rate,
        'n_lang_tokens':  n_lang,
        'n_switches':     n_switch,
        'n_alpha_tokens': n_alpha,
    }

## 5. Compute features for every master row

In [ ]:
rows = [sentence_features(t, l, VOCAB) for t, l in zip(master['tokens'], master['lang_tags'])]

feat = pd.DataFrame(rows)
feat.insert(0, 'sentence_id', master['sentence_id'].values)
feat['cmi'] = master['cmi'].values                      # carried straight through, unmodified

feat = feat[['sentence_id'] + FEATURE_COLS + DIAG_COLS]
print(feat.shape)
feat.head()

## 6. Integrity gate — non-negotiable

One row per unique `sentence_id`, no NaN, no infinities, ratios inside `[0, 1]`, `cmi` identical to the
master. If any of this fails the notebook stops here rather than writing a corrupt artifact.

In [ ]:
# --- shape / key -----------------------------------------------------------
assert len(feat) == len(master), 'row count changed vs master'
assert feat['sentence_id'].is_unique, 'sentence_id is not unique in features'
assert feat['sentence_id'].nunique() == master['sentence_id'].nunique()

# --- no missing / no infinite ---------------------------------------------
nan_counts = feat.isna().sum()
assert nan_counts.sum() == 0, f'NaNs present:\n{nan_counts[nan_counts > 0]}'
num = feat.select_dtypes(include=[np.number])
assert np.isfinite(num.to_numpy()).all(), 'non-finite values present'

# --- ranges ----------------------------------------------------------------
for c in ['switch_density', 'hi_ratio', 'caps_ratio', 'oov_rate']:
    lo, hi = feat[c].min(), feat[c].max()
    assert 0.0 <= lo and hi <= 1.0, f'{c} out of [0,1]: min={lo} max={hi}'
assert (feat['length'] >= 0).all()
assert (feat['cmi'] >= 0).all() and (feat['cmi'] <= 100).all(), 'cmi outside [0,100]'

# --- cmi really is carried, not recomputed ---------------------------------
chk = master[['sentence_id', 'cmi']].merge(feat[['sentence_id', 'cmi']], on='sentence_id',
                                           suffixes=('_master', '_feat'), validate='one_to_one')
assert len(chk) == len(master)
assert np.allclose(chk['cmi_master'], chk['cmi_feat']), 'cmi drifted from the master'

print('INTEGRITY OK —', len(feat), 'rows,', feat['sentence_id'].nunique(), 'unique sentence_id, 0 NaN')

# [ADDED] Degenerate-feature guard: a feature that is constant (zero variance) across
# the whole corpus passes every check above (no NaN, in-range) while being silently
# useless -- this is exactly how the switch_density/hi_ratio tag-mismatch bug slipped
# through undetected. Warn loudly rather than let it pass quietly.
for c in FEATURE_COLS:
    if feat[c].std() == 0:
        print(f"  !! WARNING: '{c}' has ZERO variance across all {len(feat)} sentences "
              f"(constant value {feat[c].iloc[0]}). This almost always means a bug in "
              f"how it's computed, not a real property of the data -- investigate before "
              f"proceeding to Phase 4.")


In [ ]:
feat.to_csv(FEATURES_CSV, index=False)

spec = {
    'phase': 3,
    'output': os.path.basename(FEATURES_CSV),
    'n_rows': int(len(feat)),
    'source_table': os.path.basename(MASTER_PARQUET),
    'feature_columns': FEATURE_COLS,
    'diagnostic_columns': DIAG_COLS,
    'decisions': {
        'switch_density_normalizer': SWITCH_DENSITY_NORM,
        'switch_density_formula': 'n_switches / max(1, n_lang_tokens - 1)'
                                  if SWITCH_DENSITY_NORM == 'lang_pairs'
                                  else 'n_switches / max(1, len(tokens))',
        'oov_vocab': VOCAB_PROV,
        'oov_vocab_audit': VOCAB_AUDIT,
        'oov_denominator': 'alphabetic tokens only (token.isalpha()); URLs/mentions/hashtags/emoji excluded',
        'oov_known_confound': ('romanized Hindi, slang and misspellings all count as OOV; oov_rate measures '
                               'non-canonical-English density and correlates with hi_ratio — check VIF in Phase 4 '
                               'and do not enter oov_rate and hi_ratio in the same regression untested'),
        'caps_ratio': 'fraction of alphabetic tokens with len >= %d and token.isupper()' % MIN_CAPS_LEN,
        'hi_ratio': 'count(Hin) / max(1, count(Hin) + count(Eng))',
        'cmi': 'carried unchanged from sa_master.parquet (Das & Gamback 2014)',
        'empty_denominator_policy': 'features with an empty denominator are 0.0, never NaN',
    },
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
}
with open(SPEC_JSON, 'w') as f:
    json.dump(spec, f, indent=2)

print('wrote', FEATURES_CSV)
print('wrote', SPEC_JSON)

## 7. Join onto the predictions

`left`-join predictions ← features on `sentence_id`. Row count must be preserved exactly and **zero**
rows may come back unmatched. This is the join the whole of Phase 4 rests on.

In [ ]:
if not os.path.exists(PREDS_LONG):
    print('SKIP: %s not found — the sweep is not consolidated yet.' % PREDS_LONG)
    print('      sentence_features.csv is already written and valid; re-run this cell after Phase 2.4.')
    merged = None
else:
    preds = pd.read_csv(PREDS_LONG)
    n_before = len(preds)

    need = {'sentence_id', 'model', 'seed', 'correct'}
    missing_cols = need - set(preds.columns)
    assert not missing_cols, f'predictions table is missing columns: {missing_cols}'

    # duplicate (model, seed, sentence_id) rows would silently inflate every Phase 4 statistic
    dup = int(preds.duplicated(subset=['model', 'seed', 'sentence_id']).sum())
    assert dup == 0, f'{dup} duplicate (model, seed, sentence_id) rows in {PREDS_LONG} — ' \
                     'all_predictions_long.csv was probably appended to twice'

    print('predictions rows :', n_before)
    print('models           :', sorted(preds['model'].unique()))
    print('seeds            :', sorted(preds['seed'].unique()))

    # dtype guard: a str-vs-int sentence_id mismatch would silently match nothing
    if preds['sentence_id'].dtype != feat['sentence_id'].dtype:
        print('!! sentence_id dtype mismatch (%s vs %s) — casting both to str'
              % (preds['sentence_id'].dtype, feat['sentence_id'].dtype))
        preds = preds.assign(sentence_id=preds['sentence_id'].astype(str))
        feat_join = feat.assign(sentence_id=feat['sentence_id'].astype(str))
    else:
        feat_join = feat

    orphans = set(preds['sentence_id']) - set(feat_join['sentence_id'])
    assert not orphans, f'{len(orphans)} prediction sentence_ids missing from features, e.g. {list(orphans)[:5]}'

    merged = preds.merge(feat_join, on='sentence_id', how='left', validate='many_to_one')

    assert len(merged) == n_before, f'row count changed: {n_before} -> {len(merged)}'
    unmatched = int(merged[FEATURE_COLS].isna().any(axis=1).sum())
    assert unmatched == 0, f'{unmatched} rows have NaN features after join'

    n_runs = preds.groupby(['model', 'seed']).ngroups
    per_run = preds.groupby(['model', 'seed']).size().unique()
    print('runs             :', n_runs, '| rows per run:', per_run)
    print('unique test ids  :', preds['sentence_id'].nunique())

    merged.to_csv(MERGED_CSV, index=False)
    print('JOIN OK — wrote', MERGED_CSV, merged.shape)

## 8. Sanity checks

In [ ]:
pd.set_option('display.width', 140)
print('=== feature distributions (all %d sentences) ===' % len(feat))
print(feat[FEATURE_COLS].describe().T.to_string(float_format=lambda x: '%.4f' % x))
print()
print('=== diagnostics ===')
print(feat[DIAG_COLS].describe().T.to_string(float_format=lambda x: '%.4f' % x))
print()
print('sentences with 0 language tokens :', int((feat['n_lang_tokens'] == 0).sum()))
print('sentences with 0 alpha tokens    :', int((feat['n_alpha_tokens'] == 0).sum()))
print('monolingual (switch_density==0)  :', int((feat['switch_density'] == 0).sum()))

In [ ]:
corr = feat[FEATURE_COLS].corr(method='pearson')
print('=== Pearson correlation among features ===')
print(corr.to_string(float_format=lambda x: '%+.3f' % x))
print()

# expectation: cmi / switch_density / hi_ratio / oov_rate positively related
for a, b in [('cmi', 'switch_density'), ('cmi', 'hi_ratio'), ('hi_ratio', 'oov_rate'),
             ('cmi', 'oov_rate'), ('length', 'cmi')]:
    r = corr.loc[a, b]
    flag = ''
    if abs(r) > 0.9:
        flag = '   <-- near-collinear, Phase 4 must not put both in one regression untested'
    elif a != 'length' and r < 0:
        flag = '   <-- unexpected sign, inspect'
    print('r(%-14s, %-14s) = %+.3f%s' % (a, b, r, flag))

In [ ]:
# worked examples: lowest CMI, median CMI, highest CMI — eyeball these before trusting the table
_cols = [c for c in ['sentence_id', 'split', 'label', 'text', 'tokens', 'lang_tags'] if c in master.columns]
show = (master[_cols]
        .merge(feat, on='sentence_id', validate='one_to_one')
        .sort_values('cmi')
        .reset_index(drop=True))

for i in [0, len(show) // 2, len(show) - 1]:
    r = show.iloc[i]
    print('=' * 100)
    print('sentence_id :', r['sentence_id'], '| split:', r.get('split', 'n/a'), '| label:', r.get('label', 'n/a'))
    print('text        :', str(r['text'])[:300])
    print('tokens      :', list(r['tokens'])[:40])
    print('lang_tags   :', list(r['lang_tags'])[:40])
    print('features    : cmi=%.2f  length=%d  switch_density=%.3f  hi_ratio=%.3f  caps_ratio=%.3f  oov_rate=%.3f'
          % (r['cmi'], r['length'], r['switch_density'], r['hi_ratio'], r['caps_ratio'], r['oov_rate']))
    print('              n_lang=%d  n_switches=%d  n_alpha=%d'
          % (r['n_lang_tokens'], r['n_switches'], r['n_alpha_tokens']))
print('=' * 100)

In [ ]:
# a first peek at the Phase 4 headline, if the join happened:
# accuracy by CMI tertile, per model (no CIs yet — that is Phase 4's job)
_merged = globals().get('merged', None)

if _merged is None:
    print('join skipped — no preview available')
else:
    m = _merged.copy()
    try:
        m['cmi_bucket'] = pd.qcut(m['cmi'], 3, labels=['low', 'mid', 'high'])
    except ValueError:
        # non-unique bin edges: a large mass of tied cmi values (e.g. monolingual tweets at cmi==0)
        b = pd.qcut(m['cmi'], 3, duplicates='drop')
        n_bins = len(b.cat.categories)
        names = {3: ['low', 'mid', 'high'], 2: ['low', 'high'], 1: ['all']}.get(n_bins,
                 ['q%d' % i for i in range(n_bins)])
        m['cmi_bucket'] = b.cat.rename_categories(names)
        print('NOTE: CMI ties collapsed the tertiles into %d bucket(s): %s' % (n_bins, names))
        print('      A large mass of monolingual tweets at cmi==0 does this. Phase 4 should bucket on')
        print('      explicit CMI thresholds (e.g. ==0 / 0<cmi<=median / >median) rather than qcut,')
        print('      and report the bucket sizes alongside the gap.')

    piv = m.pivot_table(index='model', columns='cmi_bucket', values='correct',
                        aggfunc='mean', observed=False)
    sizes = m.groupby('cmi_bucket', observed=False).size()

    if {'low', 'high'}.issubset(set(piv.columns)):
        piv['gap (low-high)'] = piv['low'] - piv['high']

    print('=== accuracy by CMI tertile, pooled over seeds (PREVIEW ONLY) ===')
    print('bucket sizes (rows):', dict(sizes))
    print(piv.to_string(float_format=lambda x: '%.4f' % x))
    print()
    print('Phase 4 replaces this with bootstrap CIs, per-seed variance, and significance testing.')

## Done — handoff to Phase 4

Written to Drive:

- `features/sentence_features.csv` — one row per `sentence_id`, columns
  `sentence_id, cmi, length, switch_density, hi_ratio, caps_ratio, oov_rate` + 3 diagnostics
- `features/predictions_with_features.csv` — `all_predictions_long.csv` joined to the above; the table
  Phase 4 works from directly
- `features/feature_spec.json` — the pinned decisions and vocab provenance, for the Phase 6 repo

**Carry into Phase 4:**
1. `oov_rate` and `hi_ratio` overlap by construction — run the VIF check *before* the logistic
   regression, and keep `hi_ratio` out of `correct ~ cmi + switch_density + length + oov_rate` unless
   VIF says it is safe.
2. `switch_density` is normalized by language-token pairs, not by `length`, so it is largely
   independent of `length` — the length control in the regression is doing work on `cmi` and
   `oov_rate`, not on it.
3. The diagnostic columns (`n_lang_tokens`, `n_switches`, `n_alpha_tokens`) are for auditing only.
   Do not feed them to the regression; they are deterministic functions of the features already there.